In [ ]:
# Santiago Talero Parra - 20202107025 - Modelamiento Físico Computacional

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simpson
import ipywidgets as widgets
from IPython.display import display, clear_output
import traceback

# ----------------------------
# Potenciales
# ----------------------------
def V_lineal(x, alpha=1.0, x0=0.0, V0=0.0):
    """V(x) = alpha * (x - x0) + V0"""
    return alpha * (x - x0) + V0

def V_cuadratico(x, k=1.0, x0=0.0):
    """V(x) = 0.5 * k * (x - x0)**2 (oscilador armónico)"""
    return 0.5 * k * (x - x0)**2

def V_valor_absoluto(x, alpha=1.0, x0=0.0):
    """V(x) = alpha * |x - x0|"""
    return alpha * np.abs(x - x0)

def create_potential_function(potential_type, **params):
    """Crea la función de potencial según el tipo y parámetros."""
    if potential_type == "Lineal":
        return lambda x: V_lineal(x, **params)
    elif potential_type == "Cuadrático":
        return lambda x: V_cuadratico(x, **params)
    elif potential_type == "Valor Absoluto":
        return lambda x: V_valor_absoluto(x, **params)
    else:
        raise ValueError("Tipo de potencial no reconocido")


# ----------------------------
# Utilidades numéricas
# ----------------------------
def count_nodes(psi, rel_threshold=1e-6):
    """
    Cuenta nodos (cambios de signo) ignorando cruces muy pequeños por ruido.
    rel_threshold: umbral relativo respecto a max(|psi|) para considerar un cruce significativo.
    """
    psi = np.asarray(psi)
    if psi.size < 2:
        return 0
    maxabs = np.max(np.abs(psi))
    if maxabs == 0 or not np.isfinite(maxabs):
        return 0
    diff_thresh = rel_threshold * maxabs
    cross_mask = (psi[:-1] * psi[1:]) < 0
    significant = np.abs(psi[:-1] - psi[1:]) > diff_thresh
    return int(np.sum(cross_mask & significant))

def safe_simpson(y, x):
    """Simpson seguro: verifica finitos antes de integrar."""
    y = np.asarray(y)
    x = np.asarray(x)
    if not np.all(np.isfinite(y)):
        raise ValueError("safe_simpson: valores no finitos en y")
    return simpson(y, x)


# ----------------------------
# Numerov estable
# ----------------------------
def numerov_solve(E, V_func, x_grid, m=1.0, hbar=1.0, psi0=0.0, psi1=None,
                  renorm_threshold=1e200, renorm_interval=2000):
    """
    Resuelve la ecuación de Schrödinger por Numerov (integración hacia adelante).
    - x_grid: malla uniforme (verificamos).
    - psi0, psi1: condiciones en los primeros dos puntos. Si psi1 es None usamos psi1 = dx (regla empírica).
    - renorm_threshold: valor absoluto de psi que activa reescalado periódico (previene overflow).
    - renorm_interval: cada cuantos pasos comprobamos reescalado.
    Devuelve psi (array float, no normalizado).
    """
    x_grid = np.asarray(x_grid, dtype=float)
    N = len(x_grid)
    if N < 3:
        raise ValueError("x_grid demasiado corto para Numerov")
    dx = x_grid[1] - x_grid[0]
    if not np.allclose(np.diff(x_grid), dx, rtol=1e-12, atol=0.0):
        raise ValueError("numerov_solve: x_grid debe ser uniforme")
    if psi1 is None:
        # Regla empírica: psi1 ~ dx para evitar valores demasiado pequeños en mallas con unidades físicas
        psi1 = float(dx) if dx != 0 else 1e-6

    # k^2
    V_vals = V_func(x_grid)
    if not np.all(np.isfinite(V_vals)):
        raise ValueError("numerov_solve: V contiene no-finitos")
    k2 = 2.0 * m * (E - V_vals) / (hbar**2)

    psi = np.zeros(N, dtype=float)
    psi[0] = float(psi0)
    psi[1] = float(psi1)

    factor = dx * dx / 12.0

    for i in range(1, N-1):
        denom = 1.0 + factor * k2[i+1]
        # protección contra denom ~ 0 o no-finitos
        if not np.isfinite(denom) or abs(denom) < 1e-16:
            denom = 1e-16 if np.isfinite(denom) else 1e-16

        psi_ip1 = (2.0 * (1.0 - 5.0 * factor * k2[i]) * psi[i] -
                   (1.0 + factor * k2[i-1]) * psi[i-1]) / denom
        psi[i+1] = psi_ip1

        # Reescalado periódico para prevenir overflow (no cambia nodos relativos)
        if (i % renorm_interval) == 0:
            maxabs = np.max(np.abs(psi[:i+2]))
            if not np.isfinite(maxabs):
                raise ValueError("numerov_solve: psi contiene no-finitos durante integración")
            if maxabs > renorm_threshold:
                psi[:i+2] /= maxabs

    if not np.all(np.isfinite(psi)):
        raise ValueError("numerov_solve: solución contiene NaN/Inf (probable inestabilidad)")

    return psi


# ----------------------------
# Búsqueda de niveles energéticos (muestreo + bisección por nodos)
# ----------------------------
def find_energy_level(n, V_func, x_grid, m=1.0, hbar=1.0,
                      E_min=None, E_max=None,
                      tol=1e-9, max_iter=80, samples=600):
    """
    Encuentra el nivel n (n>=1) usando:
      1) Muestreo de energies para encontrar intervalo donde el número de nodos cruza (n-1).
      2) Bisección sobre ese intervalo usando el número de nodos como criterio.
    Devuelve: E_final, psi_final (normalizada)
    """
    if n < 1 or int(n) != n:
        raise ValueError("n debe ser entero positivo (1 = estado fundamental)")

    V_vals = V_func(x_grid)
    if not np.all(np.isfinite(V_vals)):
        raise ValueError("find_energy_level: V_func devolvió no-finitos en la malla")

    Vmin = np.min(V_vals)
    Vmax = np.max(V_vals)
    span = max(1.0, Vmax - Vmin)

    if E_min is None:
        E_min = Vmin - 0.5 * span
    if E_max is None:
        E_max = Vmax + 5.0 * span

    # Asegurar orden correcto
    if E_max <= E_min:
        E_max = E_min + abs(E_min) + 1.0

    Es = np.linspace(E_min, E_max, samples)
    node_counts = np.empty_like(Es, dtype=int)

    for i, E in enumerate(Es):
        try:
            psi = numerov_solve(E, V_func, x_grid, m, hbar)
            node_counts[i] = count_nodes(psi)
        except Exception:
            # Marcamos con valor alto para que no se elija ese intervalo
            node_counts[i] = 10**6

    target_nodes = n - 1

    # Buscar intervalo donde node_counts cruza target_nodes
    bracket_idx = None
    for i in range(len(Es)-1):
        a = node_counts[i]
        b = node_counts[i+1]
        if (a < target_nodes and b >= target_nodes) or (a == target_nodes and b > target_nodes):
            bracket_idx = i
            break

    if bracket_idx is None:
        # Fallback: si target_nodes aparece exactamente en alguna muestra
        matches = [i for i, v in enumerate(node_counts) if v == target_nodes]
        if matches:
            i = matches[0]
            E_low = max(E_min, Es[max(0, i-1)])
            E_high = min(E_max, Es[min(len(Es)-1, i+1)])
        else:
            raise RuntimeError(
                f"No se encontró intervalo para n={n}. Ajusta E_min/E_max, aumenta samples o aumenta N."
            )
    else:
        E_low = Es[bracket_idx]
        E_high = Es[bracket_idx+1]

    # Bisección sobre conteo de nodos
    for iter_count in range(max_iter):
        E_mid = 0.5 * (E_low + E_high)
        try:
            psi_mid = numerov_solve(E_mid, V_func, x_grid, m, hbar)
            nodes_mid = count_nodes(psi_mid)
        except Exception:
            # Si falla, hacer ajuste conservador del intervalo
            E_low = 0.9999 * E_low + 0.0001 * E_mid
            E_high = 0.9999 * E_high + 0.0001 * E_mid
            continue

        if nodes_mid < target_nodes:
            E_low = E_mid
        else:
            E_high = E_mid

        if abs(E_high - E_low) < tol:
            break

    E_final = 0.5 * (E_low + E_high)
    psi_final = numerov_solve(E_final, V_func, x_grid, m, hbar)

    # Normalizar
    norm = np.sqrt(safe_simpson(psi_final**2, x_grid))
    if not np.isfinite(norm) or norm < 1e-16:
        raise RuntimeError("Normalización fallida: norma no finita o muy pequeña.")
    psi_final = psi_final / norm

    return E_final, psi_final


# ----------------------------
# Rango de energía automático (más inteligente)
# ----------------------------
def get_energy_guess_range_from_grid(potential_type, x_grid, V_values,
                                     alpha_lineal=None, k_cuadratico=None):
    """
    Sugiere [E_min, E_max] a partir del potencial en la malla.
    """
    Vmin = np.min(V_values)
    Vmax = np.max(V_values)
    span = Vmax - Vmin if Vmax != Vmin else 1.0

    E_min = Vmin - 0.5 * abs(span)
    E_max = Vmax + 3.0 * abs(span) + 1.0
    if abs(E_max - E_min) < 1e-6:
        E_min -= 1.0
        E_max += 10.0
    return E_min, E_max


# ----------------------------
# Función de actualización / graficado (mejorada)
# ----------------------------
def update_plot(potential_type, n_level, m, hbar, N, L,
                alpha_lineal=1.0, x0_lineal=0.0,
                k_cuadratico=1.0, x0_cuadratico=0.0, alpha_absoluto=1.0, x0_absoluto=0.0):
    """
    Actualiza la gráfica para el potencial y parámetros dados.
    """
    # Crear malla espacial
    x_grid = np.linspace(-L/2, L/2, N)
    if potential_type == "Lineal":
        params = {'alpha': alpha_lineal, 'x0': x0_lineal, 'V0': 0.0}
    elif potential_type == "Cuadrático":
        params = {'k': k_cuadratico, 'x0': x0_cuadratico}
    elif potential_type == "Valor Absoluto":
        params = {'alpha': alpha_absoluto, 'x0': x0_absoluto}
    else:
        raise ValueError("Tipo de potencial no reconocido")

    V_func = create_potential_function(potential_type, **params)
    V_values = V_func(x_grid)

    # Obtener rango de energía sugerido a partir de la malla
    E_min, E_max = get_energy_guess_range_from_grid(potential_type, x_grid, V_values,
                                                    alpha_lineal=alpha_lineal,
                                                    k_cuadratico=k_cuadratico)

    # Intentar encontrar el nivel de energía
    try:
        E_n, psi_n = find_energy_level(n_level, V_func, x_grid, m, hbar,
                                       E_min=E_min, E_max=E_max, tol=1e-9,
                                       max_iter=120, samples=800)

        # Normalizamos
        norm = np.sqrt(safe_simpson(psi_n**2, x_grid))
        if norm <= 0 or not np.isfinite(norm):
            raise RuntimeError("Normalización fallida tras find_energy_level.")
        psi_n = psi_n / norm

        # Preparar la gráfica
        fig, ax = plt.subplots(figsize=(10, 6))

        V_min = np.min(V_values)
        V_max = np.max(V_values)
        V_range = V_max - V_min if V_max != V_min else 1.0

        psi_max = np.max(np.abs(psi_n))
        if psi_max <= 0:
            psi_max = 1.0

        # Escalado: si V_range es pequeño, usar una escala absoluta razonable
        denom = psi_max
        if V_range < 1e-8:
            wave_scale = 0.5 / denom
        else:
            wave_scale = 0.3 * V_range / denom

        # Graficar
        ax.plot(x_grid, E_n + psi_n * wave_scale, label=f'ψ_{n_level}(x)')
        ax.plot(x_grid, V_values, color='red', linewidth=1.6, label='V(x)')
        ax.axhline(y=E_n, color='green', linestyle='--', linewidth=1.6, label=f'E_{n_level} = {E_n:.6f}')
        ax.set_xlabel('x')
        ax.set_ylabel('Energía / V(x)')

        title = f'{potential_type} — n={n_level}, E={E_n:.6f}'
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)

        # límites verticales razonables
        ax.set_xlim(x_grid[0], x_grid[-1])
        y_top = max(V_max, E_n) * 1.3
        ax.set_ylim(min(V_min, -0.1*y_top), y_top)

        plt.tight_layout()
        plt.show()

        # Imprimir resultados
        print(f"Energía del nivel {n_level}: E = {E_n:.8f}")
        print(f"Número de nodos (psi normalizada): {count_nodes(psi_n)}")
        print(f"Rango utilizado para búsqueda: [{E_min:.6f}, {E_max:.6f}]")

    except Exception as e:
        print("Error al calcular el nivel de energía:")
        traceback.print_exc()
        print("Sugerencias:")
        print("- Aumentar el número de puntos (N).")
        print("- Ajustar la longitud (L) para potenciales no acotados (lineal, absoluto).")
        print("- Ensayar un rango de energía más amplio (E_min, E_max).")
        print("- Revisar valores de masa (m) y ħ si trabajas con unidades físicas.")


# ----------------------------
# Interfaz con ipywidgets
# ----------------------------
def create_interactive_interface():
    """Crea la interfaz interactiva completa con ipywidgets (adaptada al solver Numerov robusto)."""
    # ---------- Widgets generales ----------
    potential_dropdown = widgets.Dropdown(
        options=['Lineal', 'Cuadrático', 'Valor Absoluto'],
        value='Lineal',
        description='Potencial:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='320px')
    )

    n_slider = widgets.IntSlider(
        value=1, min=1, max=20, step=1,
        description='Nivel n:', continuous_update=False,
        style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
    )

    m_slider = widgets.FloatSlider(
        value=1.0, min=0.1, max=10.0, step=0.1,
        description='Masa (m):', continuous_update=False,
        style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
    )

    hbar_slider = widgets.FloatSlider(
        value=1.0, min=0.01, max=10.0, step=0.01,
        description='ħ:', continuous_update=False,
        style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
    )

    N_slider = widgets.IntSlider(
        value=2000, min=100, max=40000, step=100,
        description='Puntos (N):', continuous_update=False,
        style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
    )

    auto_N_checkbox = widgets.Checkbox(
        value=True,
        description='Auto ajustar N según n (recomendado)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='320px')
    )

    L_slider = widgets.FloatSlider(
        value=10.0, min=1.0, max=200.0, step=0.5,
        description='Longitud (L):', continuous_update=False,
        style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
    )

    # ---------- Widgets específicos ----------
    lineal_params = widgets.VBox([
        widgets.FloatSlider(
            value=1.0, min=0.001, max=50.0, step=0.01,
            description='α lineal:', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
        ),
        widgets.FloatSlider(
            value=0.0, min=-100.0, max=100.0, step=0.1,
            description='x0 lineal:', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
        )
    ], layout=widgets.Layout(width='340px'))

    cuadratico_params = widgets.VBox([
        widgets.FloatSlider(
            value=1.0, min=0.001, max=100.0, step=0.01,
            description='k cuadrático:', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
        ),
        widgets.FloatSlider(
            value=0.0, min=-100.0, max=100.0, step=0.1,
            description='x0 cuadrático:', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
        )
    ], layout=widgets.Layout(width='340px'))

    absoluto_params = widgets.VBox([
        widgets.FloatSlider(
            value=1.0, min=0.001, max=50.0, step=0.01,
            description='α absoluto:', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
        ),
        widgets.FloatSlider(
            value=0.0, min=-100.0, max=100.0, step=0.1,
            description='x0 absoluto:', continuous_update=False,
            style={'description_width': 'initial'}, layout=widgets.Layout(width='320px')
        )
    ], layout=widgets.Layout(width='340px'))

    # ---------- Contenedores ----------
    general_params = widgets.VBox([
        potential_dropdown, n_slider, m_slider, hbar_slider, N_slider, auto_N_checkbox, L_slider
    ], layout=widgets.Layout(width='340px'))

    # Mostrar/ocultar según potencial
    def update_parameter_display(change):
        if isinstance(change, dict):
            val = change.get('new', potential_dropdown.value)
        else:
            val = potential_dropdown.value

        lineal_params.layout.display = 'none'
        cuadratico_params.layout.display = 'none'
        absoluto_params.layout.display = 'none'

        if val == "Lineal":
            lineal_params.layout.display = 'block'
        elif val == "Cuadrático":
            cuadratico_params.layout.display = 'block'
        elif val == "Valor Absoluto":
            absoluto_params.layout.display = 'block'

    potential_dropdown.observe(update_parameter_display, names='value')
    update_parameter_display({'new': potential_dropdown.value})

    # ---------- Área de salida y botón ----------
    output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='6px', width='820px', height='560px', overflow='auto'))

    update_button = widgets.Button(
        description="Actualizar Gráfica",
        button_style='success',
        layout=widgets.Layout(width='220px')
    )

    # función de actualización que llama a update_plot
    def on_update_button_clicked(b):
        update_button.disabled = True
        with output:
            clear_output(wait=True)
            eff_N = int(N_slider.value)
            if auto_N_checkbox.value:
                suggested = int(400 * max(1, n_slider.value))
                suggested = int(np.clip(suggested, 200, 40000))
                eff_N = max(eff_N, suggested)

            try:
                update_plot(
                    potential_dropdown.value,
                    n_slider.value,
                    m_slider.value,
                    hbar_slider.value,
                    eff_N,
                    L_slider.value,
                    alpha_lineal=lineal_params.children[0].value,
                    x0_lineal=lineal_params.children[1].value,
                    k_cuadratico=cuadratico_params.children[0].value,
                    x0_cuadratico=cuadratico_params.children[1].value,
                    alpha_absoluto=absoluto_params.children[0].value,
                    x0_absoluto=absoluto_params.children[1].value
                )

                if eff_N != N_slider.value:
                    print(f"Nota: se usó N = {eff_N} (auto-ajustado para n = {n_slider.value})")

            except Exception:
                traceback.print_exc()

            finally:
                update_button.disabled = False

    update_button.on_click(on_update_button_clicked)

    # Layout final
    params_column = widgets.VBox([
        general_params,
        lineal_params,
        cuadratico_params,
        absoluto_params,
        widgets.HBox([update_button], layout=widgets.Layout(padding='6px'))
    ], layout=widgets.Layout(width='360px'))

    interface = widgets.HBox([params_column, output], layout=widgets.Layout(align_items='flex-start'))
    display(interface)

    # Llamada inicial para mostrar la gráfica
    on_update_button_clicked(None)


# Ejecutar la interfaz interactiva cuando se ejecute el script (si procede)
if __name__ == "__main__":
    create_interactive_interface()